# 集群上对胰腺癌197个bam文件进行质控的流程

## 一、所需文件

In [ ]:
/mnt/home/ygjx/chenkejin/bam_qc/Homo_sapiens_assembly38.fasta
/mnt/home/ygjx/chenkejin/bam_qc/Homo_sapiens_assembly38.fasta.fai
/mnt/home/ygjx/chenkejin/bam_qc/Homo_sapiens_assembly38.fasta.dict
# 外加197个样本的bam文件路径或软链接

## 二、配置环境

In [ ]:
conda create -n wgs_bam_qc \
  -c conda-forge -c bioconda \
  --override-channels \
  samtools mosdepth picard qualimap fastqc multiqc openjdk \
  -y

conda activate wgs_bam_qc

# 三、批量质控脚本

## 00_make_pdac_full_bam_qc_manifest.sh

In [ ]:
#!/bin/bash
set -euo pipefail

DIR1="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/02_wgs_197_bams"
DIR2="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/WGS_bam"
OUTROOT="/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC"

mkdir -p "${OUTROOT}"/{logs,slurm_logs,summary_parts,flagstat,idxstats,mosdepth,picard,qualimap,multiqc,saz}

RAW="${OUTROOT}/manifest.raw.tsv"
MANIFEST="${OUTROOT}/pdac_197_bam_manifest.tsv"
SKIPPED="${OUTROOT}/skipped_entries.tsv"
MISSING="${OUTROOT}/missing_bam.tsv"
DUP="${OUTROOT}/duplicated_samples.before_dedup.txt"

echo -e "sample\tsource\tentry_status\tsample_dir\tbam\tbai\tbam_size_bytes\tbam_mtime" > "${RAW}"
echo -e "entry\tsource\tpath\treason" > "${SKIPPED}"
echo -e "sample\tsource\tsample_dir\treason" > "${MISSING}"

scan_root() {
  local root="$1"
  local source="$2"

  find "${root}" -maxdepth 1 -mindepth 1 ! -name "DepthStatistics" -printf "%f\t%p\n" \
  | sort \
  | while IFS=$'\t' read -r sample entry; do
      if [ -L "${entry}" ] && [ ! -e "${entry}" ]; then
        echo -e "${sample}\t${source}\t${entry}\tbroken_symlink" >> "${SKIPPED}"
        continue
      fi

      if [ ! -d "${entry}" ]; then
        echo -e "${sample}\t${source}\t${entry}\tnot_sample_directory" >> "${SKIPPED}"
        continue
      fi

      if [ -L "${entry}" ]; then
        entry_status="valid_symlink_dir"
      else
        entry_status="real_dir"
      fi

      bam=$(find -L "${entry}" -maxdepth 8 -type f -name "${sample}*.bam" | sort | head -n 1 || true)
      if [ -z "${bam}" ]; then
        bam=$(find -L "${entry}" -maxdepth 8 -type f -name "*.bam" | sort | head -n 1 || true)
      fi

      if [ -z "${bam}" ]; then
        echo -e "${sample}\t${source}\t${entry}\tno_bam_found" >> "${MISSING}"
        continue
      fi

      if [ -f "${bam}.bai" ]; then
        bai="${bam}.bai"
      elif [ -f "${bam%.bam}.bai" ]; then
        bai="${bam%.bam}.bai"
      elif [ -f "${bam}.csi" ]; then
        bai="${bam}.csi"
      elif [ -f "${bam%.bam}.csi" ]; then
        bai="${bam%.bam}.csi"
      else
        bai=$(find -L "${entry}" -maxdepth 8 -type f \( -name "*.bai" -o -name "*.csi" \) | sort | head -n 1 || true)
      fi
      [ -n "${bai:-}" ] || bai="NA"

      bam_size=$(stat -c '%s' "${bam}")
      bam_mtime=$(stat -c '%y' "${bam}" | cut -d'.' -f1)

      echo -e "${sample}\t${source}\t${entry_status}\t${entry}\t${bam}\t${bai}\t${bam_size}\t${bam_mtime}" >> "${RAW}"
    done
}

scan_root "${DIR1}" "DIR1_02_wgs_197_bams"
scan_root "${DIR2}" "DIR2_WGS_bam"

tail -n +2 "${RAW}" | cut -f1 | sort | uniq -d > "${DUP}"

{
  head -n 1 "${RAW}"
  tail -n +2 "${RAW}" | sort -k1,1 -k2,2 | awk -F'\t' '!seen[$1]++'
} > "${MANIFEST}"

sample_n=$(tail -n +2 "${MANIFEST}" | wc -l)

echo "===== manifest summary ====="
echo -n "Final sample number: "
echo "${sample_n}"

echo -n "Duplicated sample names before dedup: "
wc -l < "${DUP}"

echo -n "Missing BAM entries: "
tail -n +2 "${MISSING}" | wc -l

echo -n "Skipped entries: "
tail -n +2 "${SKIPPED}" | wc -l

echo
echo "Source count:"
awk -F'\t' 'NR>1{count[$2]++} END{for (s in count) print s, count[s]}' "${MANIFEST}" | sort

echo
echo "Manifest:"
echo "${MANIFEST}"

if [ "${sample_n}" -ne 197 ]; then
  echo "ERROR: expected 197 usable samples, got ${sample_n}" >&2
  exit 1
fi


## 01_run_pdac_full_bam_qc_array.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=pdac_full_bam_qc
#SBATCH --nodes=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=48G
#SBATCH --output=/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/slurm_logs/full_bam_qc_%A_%a.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/slurm_logs/full_bam_qc_%A_%a.err

set -euo pipefail

if [ -f /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh ]; then
  source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
  conda activate wgs_bam_qc || true
fi

MANIFEST="/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv"
OUTROOT="/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC"
THREADS="${SLURM_CPUS_PER_TASK:-8}"
QUALIMAP_JAVA_MEM="${QUALIMAP_JAVA_MEM:-36G}"
PICARD_JAVA_MEM="${PICARD_JAVA_MEM:-36G}"
REF_FASTA="${REF_FASTA:-}"

mkdir -p "${OUTROOT}"/{logs,slurm_logs,summary_parts,flagstat,idxstats,mosdepth,picard,qualimap,multiqc,saz}

if [ -z "${SLURM_ARRAY_TASK_ID:-}" ]; then
  echo "ERROR: submit this script as a Slurm array job." >&2
  exit 1
fi

if [ -z "${REF_FASTA}" ]; then
  for f in \
    "/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/Homo_sapiens_assembly38.fasta" \
    "/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/reference/Homo_sapiens_assembly38.fasta" \
    "/data/wanghao/Projects/PancreaticCancer/data/hg38/Homo_sapiens_assembly38.fasta"
  do
    if [ -f "$f" ]; then
      REF_FASTA="$f"
      break
    fi
  done
fi

line_no=$((SLURM_ARRAY_TASK_ID + 1))
line=$(sed -n "${line_no}p" "${MANIFEST}")
IFS=$'\t' read -r SAMPLE SOURCE ENTRY_STATUS SAMPLE_DIR BAM BAI BAM_SIZE_BYTES BAM_MTIME <<< "${line}"

LOG="${OUTROOT}/logs/${SAMPLE}.full_bam_qc.log"
SUMMARY_PART="${OUTROOT}/summary_parts/${SAMPLE}.full_bam_qc.tsv"
FLAGSTAT="${OUTROOT}/flagstat/${SAMPLE}.flagstat.txt"
IDXSTATS="${OUTROOT}/idxstats/${SAMPLE}.idxstats.txt"
MOS_PREFIX="${OUTROOT}/mosdepth/${SAMPLE}/${SAMPLE}"
PICARD_DIR="${OUTROOT}/picard/${SAMPLE}"
QUALIMAP_DIR="${OUTROOT}/qualimap/${SAMPLE}"
SAZ_TSV="${OUTROOT}/saz/${SAMPLE}.SAZ_fullscan.tsv"

mkdir -p "$(dirname "${MOS_PREFIX}")" "${PICARD_DIR}" "${QUALIMAP_DIR}"

echo "[Sample] ${SAMPLE}" > "${LOG}"
echo "[BAM] ${BAM}" >> "${LOG}"
echo "[BAI] ${BAI}" >> "${LOG}"
echo "[REF_FASTA] ${REF_FASTA:-NA}" >> "${LOG}"
echo "[THREADS] ${THREADS}" >> "${LOG}"

have_cmd() { command -v "$1" >/dev/null 2>&1; }

PICARD_MODE="NONE"
if have_cmd picard; then
  PICARD_MODE="COMMAND"
elif [ -n "${PICARD_JAR:-}" ] && [ -f "${PICARD_JAR}" ]; then
  PICARD_MODE="JAR"
elif compgen -G "/mnt/home/ygjx/chenkejin/anaconda3/envs/wgs_bam_qc/share/picard-*/picard.jar" >/dev/null; then
  PICARD_JAR=$(compgen -G "/mnt/home/ygjx/chenkejin/anaconda3/envs/wgs_bam_qc/share/picard-*/picard.jar" | head -n 1)
  PICARD_MODE="JAR"
fi

run_picard() {
  if [ "${PICARD_MODE}" = "COMMAND" ]; then
    picard "$@"
  elif [ "${PICARD_MODE}" = "JAR" ]; then
    java -Xmx"${PICARD_JAVA_MEM}" -jar "${PICARD_JAR}" "$@"
  else
    return 127
  fi
}

set_na_metrics() {
  BWA_PG="NA"; BWA_MEM="NA"; BWA_MEM_M="NA"
  FLAG_TOTAL="NA"; FLAG_PRIMARY="NA"; FLAG_SECONDARY="NA"; FLAG_SUPPLEMENTARY="NA"; FLAG_MAPPED="NA"; FLAG_MAPPED_PCT="NA"
  FLAG_PRIMARY_MAPPED="NA"; FLAG_PRIMARY_MAPPED_PCT="NA"; FLAG_PROPERLY_PAIRED="NA"; FLAG_PROPERLY_PAIRED_PCT="NA"
  FLAG_DUPLICATES="NA"; FLAG_DUPLICATE_PCT="NA"; FLAG_PRIMARY_DUPLICATES="NA"; FLAG_SINGLETONS="NA"; FLAG_SINGLETONS_PCT="NA"
  IDX_TOTAL_MAPPED="NA"; IDX_TOTAL_UNMAPPED="NA"; CHR_X_MAPPED="NA"; CHR_Y_MAPPED="NA"; CHR_M_MAPPED="NA"
  MOS_TOTAL_LENGTH="NA"; MOS_TOTAL_BASES="NA"; MOS_MEAN_DEPTH="NA"; MOS_MIN_DEPTH="NA"; MOS_MAX_DEPTH="NA"
  PICARD_PF_READS="NA"; PICARD_PCT_PF_READS_ALIGNED="NA"; PICARD_MEAN_READ_LENGTH="NA"
  PICARD_MEDIAN_INSERT_SIZE="NA"; PICARD_MEAN_INSERT_SIZE="NA"; PICARD_SD_INSERT_SIZE="NA"
  QUALIMAP_READS="NA"; QUALIMAP_MAPPED_READS="NA"; QUALIMAP_MEAN_COVERAGE="NA"; QUALIMAP_MEAN_MAPPING_QUALITY="NA"
  SAZ_TOTAL="NA"; SAZ_PRIMARY="NA"; SAZ_SECONDARY="NA"; SAZ_SUPPLEMENTARY="NA"; SAZ_RECORDS="NA"; SAZ_PCT="NA"
  SAZ_PRIMARY_RECORDS="NA"; SAZ_SECONDARY_RECORDS="NA"; SAZ_SUPPLEMENTARY_RECORDS="NA"
}

write_summary() {
  local conclusion="$1"
  echo -e "${SAMPLE}\t${SOURCE}\t${SAMPLE_DIR}\t${BAM}\t${BAI}\t${BAM_SIZE_BYTES}\t${BAM_MTIME}\t${QUICKCHECK_STATUS}\t${FLAGSTAT_STATUS}\t${IDXSTATS_STATUS}\t${MOSDEPTH_STATUS}\t${PICARD_ALIGNMENT_STATUS}\t${PICARD_INSERT_STATUS}\t${QUALIMAP_STATUS}\t${SAZ_STATUS}\t${BWA_PG}\t${BWA_MEM}\t${BWA_MEM_M}\t${FLAG_TOTAL}\t${FLAG_PRIMARY}\t${FLAG_SECONDARY}\t${FLAG_SUPPLEMENTARY}\t${FLAG_MAPPED}\t${FLAG_MAPPED_PCT}\t${FLAG_PRIMARY_MAPPED}\t${FLAG_PRIMARY_MAPPED_PCT}\t${FLAG_PROPERLY_PAIRED}\t${FLAG_PROPERLY_PAIRED_PCT}\t${FLAG_DUPLICATES}\t${FLAG_DUPLICATE_PCT}\t${FLAG_PRIMARY_DUPLICATES}\t${FLAG_SINGLETONS}\t${FLAG_SINGLETONS_PCT}\t${IDX_TOTAL_MAPPED}\t${IDX_TOTAL_UNMAPPED}\t${CHR_X_MAPPED}\t${CHR_Y_MAPPED}\t${CHR_M_MAPPED}\t${MOS_TOTAL_LENGTH}\t${MOS_TOTAL_BASES}\t${MOS_MEAN_DEPTH}\t${MOS_MIN_DEPTH}\t${MOS_MAX_DEPTH}\t${PICARD_PF_READS}\t${PICARD_PCT_PF_READS_ALIGNED}\t${PICARD_MEAN_READ_LENGTH}\t${PICARD_MEDIAN_INSERT_SIZE}\t${PICARD_MEAN_INSERT_SIZE}\t${PICARD_SD_INSERT_SIZE}\t${QUALIMAP_READS}\t${QUALIMAP_MAPPED_READS}\t${QUALIMAP_MEAN_COVERAGE}\t${QUALIMAP_MEAN_MAPPING_QUALITY}\t${SAZ_RECORDS}\t${SAZ_PCT}\t${SAZ_PRIMARY_RECORDS}\t${SAZ_SECONDARY_RECORDS}\t${SAZ_SUPPLEMENTARY_RECORDS}\t${conclusion}" > "${SUMMARY_PART}"
}

set_na_metrics
QUICKCHECK_STATUS="NA"; FLAGSTAT_STATUS="NA"; IDXSTATS_STATUS="NA"; MOSDEPTH_STATUS="NA"
PICARD_ALIGNMENT_STATUS="NA"; PICARD_INSERT_STATUS="NA"; QUALIMAP_STATUS="NA"; SAZ_STATUS="NA"

if [ ! -f "${BAM}" ]; then
  QUICKCHECK_STATUS="BAM_NOT_FOUND"
  write_summary "BAM_NOT_FOUND"
  exit 0
fi

if samtools quickcheck -v "${BAM}" >> "${LOG}" 2>&1; then
  QUICKCHECK_STATUS="PASS"
else
  QUICKCHECK_STATUS="FAIL"
  write_summary "QUICKCHECK_FAIL"
  exit 0
fi

HEADER_INFO=$(samtools view -H "${BAM}" | awk '
BEGIN{bwa=0; bwamem=0; minusM=0}
$0 ~ /^@PG/ && ($0 ~ /PN:bwa/ || $0 ~ /ID:bwa/) {bwa=1}
$0 ~ /^@PG/ && $0 ~ /CL:.*bwa mem/ {bwamem=1}
$0 ~ /^@PG/ && $0 ~ /CL:.*bwa mem/ && $0 ~ /(^|[ \t])-M([ \t]|$)/ {minusM=1}
END{print (bwa?"YES":"NO") "\t" (bwamem?"YES":"NO") "\t" (minusM?"YES":"NO")}
')
IFS=$'\t' read -r BWA_PG BWA_MEM BWA_MEM_M <<< "${HEADER_INFO}"

if samtools flagstat -@ "${THREADS}" "${BAM}" > "${FLAGSTAT}" 2>> "${LOG}"; then
  FLAGSTAT_STATUS="PASS"
  FLAG_METRICS=$(awk '
  BEGIN{OFS="\t"; total=primary=secondary=supplementary=mapped=primary_mapped=properly_paired=duplicates=primary_duplicates=singletons="NA"; mapped_pct=primary_mapped_pct=properly_paired_pct=duplicate_pct=singletons_pct="NA"}
  function getpct(line,s){s=line; sub(/^.*\(/,"",s); sub(/%.*$/,"",s); return (s ~ /^[0-9.]+$/ ? s : "NA")}
  / in total /{total=$1}
  / primary$/{primary=$1}
  / secondary$/{secondary=$1}
  / supplementary$/{supplementary=$1}
  / mapped \(/ && $0 !~ /primary mapped/{mapped=$1; mapped_pct=getpct($0)}
  /primary mapped/{primary_mapped=$1; primary_mapped_pct=getpct($0)}
  /properly paired/{properly_paired=$1; properly_paired_pct=getpct($0)}
  / duplicates$/ && $0 !~ /primary duplicates/{duplicates=$1}
  /primary duplicates/{primary_duplicates=$1}
  /singletons \(/{singletons=$1; singletons_pct=getpct($0)}
  END{
    if(duplicates ~ /^[0-9]+$/ && total ~ /^[0-9]+$/ && total>0) duplicate_pct=sprintf("%.6f", duplicates/total*100)
    print total,primary,secondary,supplementary,mapped,mapped_pct,primary_mapped,primary_mapped_pct,properly_paired,properly_paired_pct,duplicates,duplicate_pct,primary_duplicates,singletons,singletons_pct
  }' "${FLAGSTAT}")
  IFS=$'\t' read -r FLAG_TOTAL FLAG_PRIMARY FLAG_SECONDARY FLAG_SUPPLEMENTARY FLAG_MAPPED FLAG_MAPPED_PCT FLAG_PRIMARY_MAPPED FLAG_PRIMARY_MAPPED_PCT FLAG_PROPERLY_PAIRED FLAG_PROPERLY_PAIRED_PCT FLAG_DUPLICATES FLAG_DUPLICATE_PCT FLAG_PRIMARY_DUPLICATES FLAG_SINGLETONS FLAG_SINGLETONS_PCT <<< "${FLAG_METRICS}"
else
  FLAGSTAT_STATUS="FAIL"
fi

if samtools idxstats "${BAM}" > "${IDXSTATS}" 2>> "${LOG}"; then
  IDXSTATS_STATUS="PASS"
  IDX_METRICS=$(awk -F'\t' '
  BEGIN{OFS="\t"}
  {tm+=$3; tu+=$4}
  $1=="chrX" || $1=="X"{x+=$3}
  $1=="chrY" || $1=="Y"{y+=$3}
  $1=="chrM" || $1=="MT" || $1=="M"{m+=$3}
  END{print tm+0,tu+0,x+0,y+0,m+0}' "${IDXSTATS}")
  IFS=$'\t' read -r IDX_TOTAL_MAPPED IDX_TOTAL_UNMAPPED CHR_X_MAPPED CHR_Y_MAPPED CHR_M_MAPPED <<< "${IDX_METRICS}"
else
  IDXSTATS_STATUS="FAIL"
fi

if have_cmd mosdepth; then
  if mosdepth -t "${THREADS}" -n "${MOS_PREFIX}" "${BAM}" >> "${LOG}" 2>&1; then
    MOSDEPTH_STATUS="PASS"
    MOS_SUMMARY="${MOS_PREFIX}.mosdepth.summary.txt"
    if [ -f "${MOS_SUMMARY}" ]; then
      MOS_METRICS=$(awk 'BEGIN{OFS="\t"} $1=="total"{print $2,$3,$4,$5,$6; found=1} END{if(!found) print "NA\tNA\tNA\tNA\tNA"}' "${MOS_SUMMARY}")
      IFS=$'\t' read -r MOS_TOTAL_LENGTH MOS_TOTAL_BASES MOS_MEAN_DEPTH MOS_MIN_DEPTH MOS_MAX_DEPTH <<< "${MOS_METRICS}"
    fi
  else
    MOSDEPTH_STATUS="FAIL"
  fi
else
  MOSDEPTH_STATUS="MISSING_TOOL"
fi

if [ "${PICARD_MODE}" = "NONE" ]; then
  PICARD_ALIGNMENT_STATUS="MISSING_TOOL"
  PICARD_INSERT_STATUS="MISSING_TOOL"
else
  PICARD_INSERT_METRICS="${PICARD_DIR}/${SAMPLE}.insert_size_metrics.txt"
  PICARD_INSERT_PDF="${PICARD_DIR}/${SAMPLE}.insert_size_histogram.pdf"
  if run_picard CollectInsertSizeMetrics I="${BAM}" O="${PICARD_INSERT_METRICS}" H="${PICARD_INSERT_PDF}" M=0.5 VALIDATION_STRINGENCY=SILENT >> "${LOG}" 2>&1; then
    PICARD_INSERT_STATUS="PASS"
    PICARD_INSERT_VALUES=$(awk '
    /^MEDIAN_INSERT_SIZE\t/{header=$0; getline; data=$0; split(header,h,"\t"); split(data,d,"\t"); for(i=1;i<=length(h);i++) v[h[i]]=d[i]; print v["MEDIAN_INSERT_SIZE"] "\t" v["MEAN_INSERT_SIZE"] "\t" v["STANDARD_DEVIATION"]; exit}
    END{if(!header) print "NA\tNA\tNA"}' "${PICARD_INSERT_METRICS}")
    IFS=$'\t' read -r PICARD_MEDIAN_INSERT_SIZE PICARD_MEAN_INSERT_SIZE PICARD_SD_INSERT_SIZE <<< "${PICARD_INSERT_VALUES}"
  else
    PICARD_INSERT_STATUS="FAIL"
  fi

  PICARD_ALIGNMENT_METRICS="${PICARD_DIR}/${SAMPLE}.alignment_summary_metrics.txt"
  if [ -n "${REF_FASTA}" ] && [ -f "${REF_FASTA}" ]; then
    if run_picard CollectAlignmentSummaryMetrics R="${REF_FASTA}" I="${BAM}" O="${PICARD_ALIGNMENT_METRICS}" VALIDATION_STRINGENCY=SILENT >> "${LOG}" 2>&1; then
      PICARD_ALIGNMENT_STATUS="PASS"
      PICARD_ALIGNMENT_VALUES=$(awk '
      /^CATEGORY\t/{header=$0; next}
      header && $1=="PAIR"{data=$0; split(header,h,"\t"); split(data,d,"\t"); for(i=1;i<=length(h);i++) v[h[i]]=d[i]; print v["PF_READS"] "\t" v["PCT_PF_READS_ALIGNED"] "\t" v["MEAN_READ_LENGTH"]; found=1; exit}
      END{if(!found) print "NA\tNA\tNA"}' "${PICARD_ALIGNMENT_METRICS}")
      IFS=$'\t' read -r PICARD_PF_READS PICARD_PCT_PF_READS_ALIGNED PICARD_MEAN_READ_LENGTH <<< "${PICARD_ALIGNMENT_VALUES}"
    else
      PICARD_ALIGNMENT_STATUS="FAIL"
    fi
  else
    PICARD_ALIGNMENT_STATUS="SKIP_NO_REFERENCE"
  fi
fi

if have_cmd qualimap; then
  if qualimap bamqc -bam "${BAM}" -outdir "${QUALIMAP_DIR}" -nt "${THREADS}" --java-mem-size="${QUALIMAP_JAVA_MEM}" >> "${LOG}" 2>&1; then
    QUALIMAP_STATUS="PASS"
    QUALIMAP_RESULTS="${QUALIMAP_DIR}/genome_results.txt"
    if [ -f "${QUALIMAP_RESULTS}" ]; then
      QUALIMAP_VALUES=$(awk -F'= ' '
      function clean(x){gsub(/^[ \t]+|[ \t]+$/,"",x); gsub(/,/,"",x); gsub(/X$/,"",x); return x}
      /number of reads =/{reads=clean($2)}
      /number of mapped reads =/{mapped=clean($2)}
      /mean coverageData =/{cov=clean($2)}
      /mean mapping quality =/{mapq=clean($2)}
      END{print (reads?reads:"NA") "\t" (mapped?mapped:"NA") "\t" (cov?cov:"NA") "\t" (mapq?mapq:"NA")}' "${QUALIMAP_RESULTS}")
      IFS=$'\t' read -r QUALIMAP_READS QUALIMAP_MAPPED_READS QUALIMAP_MEAN_COVERAGE QUALIMAP_MEAN_MAPPING_QUALITY <<< "${QUALIMAP_VALUES}"
    fi
  else
    QUALIMAP_STATUS="FAIL"
  fi
else
  QUALIMAP_STATUS="MISSING_TOOL"
fi

if SAZ_METRICS=$(samtools view -@ "${THREADS}" "${BAM}" 2>> "${LOG}" | awk '
BEGIN{OFS="\t"}
{
  total++
  flag=$2+0
  sec=int(flag/256)%2
  supp=int(flag/2048)%2
  pri=(!sec && !supp)
  if(pri) primary++
  if(sec) secondary++
  if(supp) supplementary++
  has_sa=0
  for(i=12;i<=NF;i++){
    if($i ~ /^SA:Z:/){has_sa=1; break}
  }
  if(has_sa){
    saz++
    if(pri) saz_primary++
    if(sec) saz_secondary++
    if(supp) saz_supplementary++
  }
}
END{
  saz_pct=(total>0 ? sprintf("%.6f", saz/total*100) : "NA")
  print total+0,primary+0,secondary+0,supplementary+0,saz+0,saz_pct,saz_primary+0,saz_secondary+0,saz_supplementary+0
}'); then
  SAZ_STATUS="PASS"
  echo -e "total_records\tprimary_records\tsecondary_records\tsupplementary_records\tSAZ_records\tSAZ_pct\tSAZ_primary_records\tSAZ_secondary_records\tSAZ_supplementary_records" > "${SAZ_TSV}"
  echo -e "${SAZ_METRICS}" >> "${SAZ_TSV}"
  IFS=$'\t' read -r SAZ_TOTAL SAZ_PRIMARY SAZ_SECONDARY SAZ_SUPPLEMENTARY SAZ_RECORDS SAZ_PCT SAZ_PRIMARY_RECORDS SAZ_SECONDARY_RECORDS SAZ_SUPPLEMENTARY_RECORDS <<< "${SAZ_METRICS}"
else
  SAZ_STATUS="FAIL"
fi

CONCLUSION="CHECK"
if [ "${FLAG_SUPPLEMENTARY}" = "0" ] && [ "${SAZ_RECORDS}" != "NA" ] && [ "${SAZ_RECORDS}" -gt 0 ]; then
  CONCLUSION="SUPP_ZERO_SAZ_PRESENT"
elif [ "${FLAG_SUPPLEMENTARY}" = "0" ] && [ "${SAZ_RECORDS}" = "0" ]; then
  CONCLUSION="SUPP_ZERO_NO_SAZ"
elif [ "${FLAG_SUPPLEMENTARY}" != "NA" ] && [ "${FLAG_SUPPLEMENTARY}" -gt 0 ]; then
  CONCLUSION="SUPP_PRESENT"
fi

write_summary "${CONCLUSION}"
echo "Done: ${SAMPLE}" >> "${LOG}"


## 02_summarize_pdac_full_bam_qc.sh

In [ ]:
#!/bin/bash
set -euo pipefail

OUTROOT="/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC"
SUMMARY="${OUTROOT}/all_samples.full_BAM_QC.summary.tsv"
KEY_SUMMARY="${OUTROOT}/all_samples.full_BAM_QC.key_summary.tsv"
FAILED="${OUTROOT}/samples_failed_any_QC.tsv"
SUPP_ZERO_SAZ="${OUTROOT}/samples_supplementary_zero_but_SAZ_present.tsv"
SUPP_ZERO_NO_SAZ="${OUTROOT}/samples_supplementary_zero_and_no_SAZ.tsv"
CONCLUSION_COUNT="${OUTROOT}/conclusion_counts.tsv"
TOOL_STATUS_COUNT="${OUTROOT}/tool_status_counts.tsv"
MULTIQC_DIR="${OUTROOT}/multiqc"

mkdir -p "${MULTIQC_DIR}"

HEADER="sample\tsource\tsample_dir\tbam\tbai\tbam_size_bytes\tbam_mtime\tquickcheck_status\tflagstat_status\tidxstats_status\tmosdepth_status\tpicard_alignment_status\tpicard_insert_status\tqualimap_status\tSAZ_status\tbwa_pg\tbwa_mem\tbwa_mem_M\tflagstat_total_reads\tflagstat_primary_reads\tflagstat_secondary_reads\tflagstat_supplementary_reads\tflagstat_mapped_reads\tflagstat_mapped_pct\tflagstat_primary_mapped_reads\tflagstat_primary_mapped_pct\tflagstat_properly_paired_reads\tflagstat_properly_paired_pct\tflagstat_duplicate_reads\tflagstat_duplicate_pct\tflagstat_primary_duplicate_reads\tflagstat_singletons\tflagstat_singletons_pct\tidxstats_total_mapped\tidxstats_total_unmapped\tidxstats_chrX_mapped\tidxstats_chrY_mapped\tidxstats_chrM_mapped\tmosdepth_total_length\tmosdepth_total_bases\tmosdepth_mean_depth\tmosdepth_min_depth\tmosdepth_max_depth\tpicard_PF_READS\tpicard_PCT_PF_READS_ALIGNED\tpicard_MEAN_READ_LENGTH\tpicard_MEDIAN_INSERT_SIZE\tpicard_MEAN_INSERT_SIZE\tpicard_INSERT_SIZE_SD\tqualimap_number_of_reads\tqualimap_number_of_mapped_reads\tqualimap_mean_coverage\tqualimap_mean_mapping_quality\tSAZ_records\tSAZ_pct\tSAZ_primary_records\tSAZ_secondary_records\tSAZ_supplementary_records\tconclusion"

echo -e "${HEADER}" > "${SUMMARY}"
find "${OUTROOT}/summary_parts" -type f -name "*.full_bam_qc.tsv" | sort | xargs -r cat >> "${SUMMARY}"

awk -F'\t' 'NR==1 || $8!="PASS" || $9!="PASS" || $10!="PASS" || $11!="PASS" || $12!="PASS" || $13!="PASS" || $14!="PASS" || $15!="PASS"' "${SUMMARY}" > "${FAILED}"
awk -F'\t' 'NR==1 || ($22==0 && $54>0)' "${SUMMARY}" > "${SUPP_ZERO_SAZ}"
awk -F'\t' 'NR==1 || ($22==0 && $54==0)' "${SUMMARY}" > "${SUPP_ZERO_NO_SAZ}"

{
  echo -e "conclusion\tcount"
  awk -F'\t' 'NR>1{count[$59]++} END{for (c in count) print c "\t" count[c]}' "${SUMMARY}" | sort
} > "${CONCLUSION_COUNT}"

{
  echo -e "tool\tstatus\tcount"
  awk -F'\t' '
  NR>1{
    tools[8]="quickcheck"; tools[9]="flagstat"; tools[10]="idxstats"; tools[11]="mosdepth"; tools[12]="picard_alignment"; tools[13]="picard_insert"; tools[14]="qualimap"; tools[15]="SAZ"
    for(i=8;i<=15;i++) count[tools[i] "\t" $i]++
  }
  END{for(k in count) print k "\t" count[k]}' "${SUMMARY}" | sort
} > "${TOOL_STATUS_COUNT}"

awk -F'\t' '
BEGIN{OFS="\t"}
NR==1{
  print "sample","source","quickcheck","flagstat","idxstats","mosdepth","picard_alignment","picard_insert","qualimap","SAZ_status","bwa_mem_M","total_reads","mapped_pct","properly_paired_pct","duplicate_pct","secondary_reads","supplementary_reads","SAZ_records","SAZ_pct","SAZ_secondary_records","mosdepth_mean_depth","picard_mean_insert_size","qualimap_mean_coverage","chrX_mapped","chrY_mapped","chrM_mapped","conclusion"
  next
}
{
  print $1,$2,$8,$9,$10,$11,$12,$13,$14,$15,$18,$19,$24,$28,$30,$21,$22,$54,$55,$57,$41,$48,$52,$36,$37,$38,$59
}' "${SUMMARY}" > "${KEY_SUMMARY}"

if command -v multiqc >/dev/null 2>&1; then
  multiqc "${OUTROOT}" -o "${MULTIQC_DIR}" -n "PDAC_197_full_BAM_QC_multiqc.html" -f
  MULTIQC_STATUS="PASS"
else
  MULTIQC_STATUS="MISSING_TOOL"
fi

echo "===== full BAM QC summary ====="
echo -n "Total summarized samples: "
tail -n +2 "${SUMMARY}" | wc -l

echo -n "Samples failed at least one required QC step: "
tail -n +2 "${FAILED}" | wc -l

echo -n "bwa mem -M detected samples: "
awk -F'\t' 'NR>1 && $18=="YES"{n++} END{print n+0}' "${SUMMARY}"

echo -n "supplementary=0 samples: "
awk -F'\t' 'NR>1 && $22==0{n++} END{print n+0}' "${SUMMARY}"

echo -n "supplementary=0 but SA:Z present samples: "
awk -F'\t' 'NR>1 && $22==0 && $54>0{n++} END{print n+0}' "${SUMMARY}"

echo -n "supplementary=0 and no SA:Z samples: "
awk -F'\t' 'NR>1 && $22==0 && $54==0{n++} END{print n+0}' "${SUMMARY}"

echo
echo "MultiQC status: ${MULTIQC_STATUS}"
echo
echo "Tool status counts:"
cat "${TOOL_STATUS_COUNT}"

echo
echo "Conclusion counts:"
cat "${CONCLUSION_COUNT}"

echo
echo "Main summary:"
echo "${SUMMARY}"
echo
echo "Key summary:"
echo "${KEY_SUMMARY}"
echo
echo "MultiQC report:"
echo "${MULTIQC_DIR}/PDAC_197_full_BAM_QC_multiqc.html"


# 三、运行

In [2]:
cd /mnt/home/ygjx/chenkejin/bam_qc

chmod +x 00_make_pdac_full_bam_qc_manifest.sh \
         01_run_pdac_full_bam_qc_array.sh \
         02_summarize_pdac_full_bam_qc.sh

bash 00_make_pdac_full_bam_qc_manifest.sh

# 先确认Final sample number: 197

SyntaxError: invalid decimal literal (1071064561.py, line 5)

In [ ]:
MANIFEST="/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv"
N=$(($(wc -l < "${MANIFEST}") - 1))
echo "${N}"

sbatch -p cu_share --array=1-${N}%10 /mnt/home/ygjx/chenkejin/bam_qc/01_run_pdac_full_bam_qc_array.sh

In [1]:
# 汇总
bash /mnt/home/ygjx/chenkejin/bam_qc/02_summarize_pdac_full_bam_qc.sh

SyntaxError: invalid decimal literal (2285562396.py, line 1)

# 结果

In [ ]:
bam_qc/PDAC_WGS_full_BAM_QC/all_samples.full_BAM_QC.summary.tsv